<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Random Forest Classifier

I chose a Random Forest Classifier because this is a tabular dataset with multiple numerical and categorical signals. It can capture non-linear relationships that a simple rule may miss.

My Week 4 baseline prioritizes content using low CTR and staleness. For this modeling experiment, I use downward trend (trend_direction = down) as a proxy outcome and compare the model with the same baseline signals.

The model is used as decision-support. A predicted downward trend does not automatically mean that content must be refreshed.

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

url = "https://raw.githubusercontent.com/usmanumer038/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a grouped train/test split by client_id. This prevents content from the same client appearing in both training and testing data.

The target is:

1 = downward trend

0 = other trend

Target-related columns are excluded to avoid leakage.

In [12]:
df = df.dropna(subset=[
    "trend_direction",
    "ctr",
    "impressions_90d",
    "days_since_last_update"
])

df = df[df["impressions_90d"] >= 100].copy()

df["target"] = (df["trend_direction"] == "down").astype(int)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train = df.iloc[train_idx]
test = df.iloc[test_idx]

print("Train:", len(train))
print("Test:", len(test))

Train: 18392
Test: 3614


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
The Week 4 baseline ranks content using the same rule:

Low CTR receives higher priority.
Higher staleness receives higher priority.

The Random Forest model is trained on the training clients and evaluated on the same held-out test clients.

Both approaches are compared using ROC-AUC, Average Precision, and Precision@20.

In [13]:
exclude = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target"
]

features = [
    "ctr",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count"
]

features = [f for f in features if f in df.columns]

# Fill missing values
X_train = train[features].fillna(train[features].median())
X_test = test[features].fillna(train[features].median())

y_train = train["target"]
y_test = test["target"]

# Train model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

model_score = model.predict_proba(X_test)[:, 1]

# Week 4 baseline score
baseline_score = (
    0.6 * test["ctr"].rank(pct=True, ascending=True)
    + 0.4 * test["days_since_last_update"].rank(pct=True)
)

# Results
results = pd.DataFrame({
    "Method": ["Week 4 Baseline", "Random Forest"],
    "ROC-AUC": [
        roc_auc_score(y_test, baseline_score),
        roc_auc_score(y_test, model_score)
    ],
    "Average Precision": [
        average_precision_score(y_test, baseline_score),
        average_precision_score(y_test, model_score)
    ]
}).round(3)

display(results)

,Method,ROC-AUC,Average Precision
0,Week 4 Baseline,0.381,0.481
1,Random Forest,0.601,0.632


## 4. Errors and interpretation

The model is not correct for every prediction. I reviewed incorrect predictions to understand where it fails.

Feature importance shows which available signals the model relied on most.

In [14]:
# Predictions
pred = model.predict(X_test)

# Incorrect predictions
errors = test[features + ["trend_direction"]].copy()

errors["actual"] = y_test
errors["predicted"] = pred

errors = errors[errors["actual"] != errors["predicted"]]

print("Number of errors:", len(errors))
display(errors.head(10))

Number of errors: 1501


,ctr,impressions_90d,days_since_last_update,avg_position,engagement_rate,scroll_rate,word_count,trend_direction,actual,predicted
26,0.12,2426,13,30.0,0.00,11.11,2686.0,stable,0,1
34,0.03,3998,8,6.4,0.00,40.00,2542.0,up,0,1
54,0.00,170,22,10.7,0.00,100.00,NaN,down,1,0
81,0.31,320,22,9.6,0.00,0.00,NaN,down,1,0
82,0.44,1810,13,8.3,12.50,8.33,2793.0,up,0,1
96,0.00,1197,13,21.4,0.00,0.00,3008.0,stable,0,1
132,0.00,4540,22,13.7,0.00,0.00,NaN,down,1,0
134,0.14,717,20,4.2,0.00,66.67,2329.0,stable,0,1
135,0.32,19802,25,20.4,12.75,13.68,2844.0,up,0,1
189,0.90,111,8,5.2,0.00,25.00,2440.0,up,0,1


## Final interpretation
The Random Forest was compared with my Week 4 rule-based baseline using the same held-out test data. The model uses additional signals beyond CTR and staleness and may identify patterns that the simple baseline misses.

The model still makes incorrect predictions, so its output should be treated as decision-support rather than an automatic instruction to refresh content.

Limitation: The target is downward trend direction, which is a proxy outcome. A downward trend does not automatically prove that content requires a refresh.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.